# Model Deployment & Endpoints

## Real-Time Endpoints

Real-time endpoints provide low-latency predictions for synchronous requests. They maintain warm instances ready to serve predictions immediately.

In [ ]:
from sagemaker.estimator import Estimator
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'

# Train a model
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path='s3://my-bucket/output',
    sagemaker_session=session
)

estimator.fit({'training': 's3://my-bucket/train-data/'})

# Deploy as real-time endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='xgboost-realtime-endpoint'
)

# Make predictions
import csv
import io

test_data = '5.1,3.5,1.4,0.2'
response = predictor.predict(test_data)
print(f"Prediction: {response}")

## Serverless Endpoints

Serverless endpoints automatically scale based on traffic, eliminating the need to manage instances. They're ideal for variable workloads.

In [ ]:
from sagemaker.serverless import ServerlessInferenceConfig

# Create serverless endpoint
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=1024,
    max_concurrency=10
)

predictor = estimator.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name='xgboost-serverless-endpoint'
)

# Invoke serverless endpoint
response = predictor.predict(test_data)
print(f"Prediction: {response}")

## Async Inference

Async inference handles large payloads and long-running predictions. Requests are queued and results are stored in S3.

In [ ]:
from sagemaker.async_inference.async_inference_config import AsyncInferenceConfig

# Configure async inference
async_config = AsyncInferenceConfig(
    output_path='s3://my-bucket/async-output/',
    max_concurrent_invocations_per_instance=10
)

predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    async_inference_config=async_config,
    endpoint_name='xgboost-async-endpoint'
)

# Invoke async endpoint
import json

input_location = 's3://my-bucket/async-input/test-data.json'
response = predictor.predict_async(input_location)
output_location = response.output_location
print(f"Output will be at: {output_location}")

## Multi-Model Endpoints

Multi-model endpoints host multiple models on a single endpoint, reducing costs and simplifying management.

In [ ]:
from sagemaker.multidatamodel import MultiDataModel

# Create multi-model endpoint
multi_model = MultiDataModel(
    name='multi-model-endpoint',
    model_data_prefix='s3://my-bucket/models/',
    model_name='xgboost-multi',
    container_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    sagemaker_session=session
)

# Add models
multi_model.add('model-1.tar.gz')
multi_model.add('model-2.tar.gz')

# Deploy
predictor = multi_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

# Invoke specific model
response = predictor.predict(
    test_data,
    target_model='model-1.tar.gz'
)

## Endpoint Configuration

```json
{
  "endpoint_config": {
    "endpoint_name": "my-endpoint",
    "endpoint_config_name": "my-endpoint-config",
    "production_variants": [
      {
        "variant_name": "variant-1",
        "model_name": "my-model",
        "initial_instance_count": 1,
        "instance_type": "ml.m5.large",
        "initial_variant_weight": 1.0
      }
    ],
    "data_capture_config": {
      "enabled": true,
      "initial_sampling_percentage": 100,
      "destination_s3_uri": "s3://my-bucket/data-capture/"
    }
  }
}
```

## Auto-Scaling Endpoints

In [ ]:
import boto3

autoscaling = boto3.client('application-autoscaling')

# Register endpoint for auto-scaling
autoscaling.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId='endpoint/my-endpoint/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=1,
    MaxCapacity=10
)

# Create scaling policy
autoscaling.put_scaling_policy(
    PolicyName='endpoint-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId='endpoint/my-endpoint/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 70.0,
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
        }
    }
)

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the primary advantage of real-time endpoints?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>Low-latency synchronous predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>Automatic scaling</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>Batch processing</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>Cost savings</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ When should you use serverless endpoints?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>For high-throughput, consistent traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>For variable or unpredictable traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>For batch processing</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>For real-time low-latency predictions</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is async inference best for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Real-time predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Small payloads</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Large payloads and long-running predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>Low-latency requirements</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the main benefit of multi-model endpoints?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>Host multiple models on one endpoint, reducing costs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>Faster predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Automatic model selection</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Better accuracy</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does auto-scaling do for endpoints?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>Automatically trains new models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>Adjusts instance count based on traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>Monitors model performance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>Deploys new models</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>